import all libraries

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import joblib
import torch 
import torch.nn as nn
import numpy as np

connect to db

In [9]:
load_dotenv()
db_url = os.getenv("DATABASE_URL")
engine = create_engine(db_url)
player = pd.read_sql("SELECT * FROM playerstat", engine)
league = pd.read_sql("SELECT * FROM leaguestat", engine)
league.head()

,Season,PA,RS,IP,G,1B,BB,IBB,HBP,SB,CS,GDP,GDP_OPP,league_wOBA,rpw
0,1871,11215.0,2659,2250.000000,277,2381,393,0.0,0.0,441.0,123.0,74.0,10775.0,0.312350,8.318000
1,1872,15928.0,3390,3286.000000,405,3704,263,0.0,0.0,269.0,134.0,97.0,15628.0,0.285315,7.642422
2,1873,17294.0,3580,3584.666667,434,4098,335,0.0,0.0,314.0,131.0,122.0,16912.0,0.293449,7.494142
3,1874,19342.0,3470,4169.666667,490,4356,238,0.0,0.0,242.0,97.0,107.0,19064.0,0.270910,6.744904
4,1875,27082.0,4234,6190.333333,763,5660,249,0.0,0.0,629.0,320.0,142.0,26793.0,0.250098,6.077863


Combine player dataframe and league dataframe

In [10]:
league = league.rename(columns={"Season": "season"})
combined = pd.merge(player, league, on="season")
pd.set_option('display.max_columns', None)
combined.head()

,idfg,name,season,age,pa,ab,h,1B_x,2B,3B,hr,bb,ibb,ubb,hbp,sb,cs,gdp,gdp_opp,sf,woba,PA,RS,IP,G,1B_y,BB,IBB,HBP,SB,CS,GDP,GDP_OPP,league_wOBA,rpw
0,aardsda01,David Aardsma,2004,23.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,188519.0,23376,43394.000000,18272,29254,16222,1381.0,1850.0,2589.0,1100.0,3784.0,163633.0,0.333745,5.424114
1,aardsda01,David Aardsma,2006,25.0,3.0,2,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,188052.0,23599,43258.000000,18694,29600,15847,1410.0,1817.0,2767.0,1110.0,3945.0,163606.0,0.335818,5.454933
2,aardsda01,David Aardsma,2007,26.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,188598.0,23322,43425.666667,19294,29885,16079,1323.0,1755.0,2918.0,1002.0,3983.0,164366.0,0.332051,5.416750
3,aardsda01,David Aardsma,2008,27.0,1.0,1,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,187614.0,22585,43357.666667,19012,29194,16337,1310.0,1672.0,2799.0,1035.0,3883.0,163362.0,0.328572,5.344049
4,aardsda01,David Aardsma,2009,28.0,0.0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,187060.0,22419,43272.000000,19097,28796,16620,1179.0,1590.0,2970.0,1133.0,3796.0,162442.0,0.328832,5.331427


Compute wRAA, BsR, RLR(https://www.youtube.com/watch?v=hlFIipW2Gfs)

In [11]:
Sabermetric = pd.DataFrame()
Sabermetric['IDfg'] = combined['idfg']
Sabermetric['Age'] = combined['age']
Sabermetric['Season'] = combined['season']
Sabermetric['wRAA'] = ((combined['woba'] - combined['league_wOBA'])/1.25) * combined['pa'] #Used wOBA Scale = 1.25 

Sabermetric['wGDP'] = (((combined['GDP']/combined['GDP_OPP'])*combined['gdp_opp'])-combined['gdp'])*(combined['RS']/(3*combined['IP']))
Sabermetric['runCS'] = (-2*(combined['RS']/(combined['IP']*3))) + 0.075
Sabermetric['league_wSB'] = ((combined['SB']*0.2) + (combined['CS']*Sabermetric['runCS']))/(combined['1B_y']+combined['BB']+combined['HBP']+combined['IBB'])
Sabermetric['wSB'] = ((combined['sb']*0.2) + (combined['cs']*Sabermetric['runCS']))-(Sabermetric['league_wSB']*(combined['1B_x']+combined['bb']+combined['hbp']+combined['ibb']))
Sabermetric['BsR'] = Sabermetric['wSB'] + Sabermetric['wGDP']

Sabermetric['RPW'] = (9*(combined['RS']/(combined['IP']*3))*1.5) + 3 #assume 9 innings per game
Sabermetric['RLR'] = ((0.235*2430)  * Sabermetric['RPW'] * combined['pa'])/combined['PA'] #assume full season, 162 games for 30 teams, divide by 2 to count wins and losses

Sabermetric = Sabermetric[['IDfg', 'Age', 'Season', 'wRAA', 'BsR', 'RLR', 'wGDP', 'runCS', 'league_wSB', 'wSB', 'RPW']]
Sabermetric['wRAA'] = Sabermetric['wRAA'].replace([np.inf, -np.inf], np.nan)
Sabermetric['BsR'] = Sabermetric['BsR'].replace([np.inf, -np.inf], np.nan)
Sabermetric['RLR'] = Sabermetric['RLR'].replace([np.inf, -np.inf], np.nan)
Sabermetric = Sabermetric.dropna()
Sabermetric.head()

,IDfg,Age,Season,wRAA,BsR,RLR,wGDP,runCS,league_wSB,wSB,RPW
1,aardsda01,25.0,2006,-0.805962,0.013155,0.049694,0.013155,-0.288694,0.004786,0.000000,5.454933
3,aardsda01,27.0,2008,-0.262858,0.004127,0.016266,0.004127,-0.272267,0.005731,0.000000,5.344049
8,aardsda01,34.0,2015,-0.252409,0.003664,0.015989,0.003664,-0.242102,0.005452,0.000000,5.140440
9,aaronha01,20.0,1954,-0.687195,-0.485575,15.826681,-0.352719,-0.251213,0.000262,-0.132857,5.201936
10,aaronha01,21.0,1955,35.688688,-0.715454,21.031630,-1.059813,-0.260226,-0.000027,0.344359,5.262778


Load Sabermetric onto DB

In [12]:
Sabermetric.to_sql(
    name='playersabermetric',
    con=engine,
    if_exists='replace',  
    index=False
)

319

Load leaguestat DB

In [13]:
leaguestat = pd.read_sql("SELECT * FROM leaguestat", engine)

Start spliiting data

In [14]:

X = Sabermetric[
    [
        'Age',
        'wGDP',
        'runCS',
        'league_wSB',
        'wSB',
        'RPW',
        'BsR',
        'RLR'
    ]
]
y = Sabermetric[['wRAA', 'RLR', 'BsR']]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
X_wraa_train = X_train[['Age', 'BsR', 'wSB','RLR','wGDP']]
X_wraa_test  = X_test[['Age', 'BsR', 'wSB','RLR','wGDP']]

X_rlr_train = X_train[['Age', 'RPW', 'league_wSB', 'wGDP']]
X_rlr_test  = X_test[['Age', 'RPW', 'league_wSB', 'wGDP']]

X_bsr_train = X_train[['wSB', 'runCS', 'league_wSB']]
X_bsr_test  = X_test[['wSB', 'runCS', 'league_wSB']]


Train random forest regressor

In [17]:
rf_wraa = RandomForestRegressor(n_estimators=100, random_state=42)  
rf_wraa.fit(X_wraa_train, y_train['wRAA'])
rf_rlr = RandomForestRegressor(n_estimators=100, random_state=42)  
rf_rlr.fit(X_rlr_train, y_train['RLR']) 
rf_bsr = RandomForestRegressor(n_estimators=100, random_state=42)
rf_bsr.fit(X_bsr_train, y_train['BsR'])

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

Store model

In [18]:

joblib.dump(rf_wraa, '../model/rf_wraa.pkl')
joblib.dump(rf_rlr, '../model/rf_rlr_model.pkl') 
joblib.dump(rf_bsr, '../model/rf_bsr_model.pkl')


['../model/rf_bsr_model.pkl']

Clean data for neural network training

In [19]:
scaler_X_wraa = StandardScaler()
scaler_X_rlr  = StandardScaler()
scaler_X_bsr  = StandardScaler()
scaler_y_wraa = StandardScaler()
scaler_y_bsr  = StandardScaler()
scaler_y_rlr  = StandardScaler()

X_wraa = scaler_X_wraa.fit_transform(X_wraa_train.values.astype(np.float32))
X_rlr  = scaler_X_rlr.fit_transform(X_rlr_train.values.astype(np.float32))
X_bsr  = scaler_X_bsr.fit_transform(X_bsr_train.values.astype(np.float32))

y_wraa_scaled = scaler_y_wraa.fit_transform(y_train[["wRAA"]].values.astype(np.float32))
y_bsr_scaled  = scaler_y_bsr.fit_transform(y_train[["BsR"]].values.astype(np.float32))
y_rlr_scaled  = scaler_y_rlr.fit_transform(y_train[["RLR"]].values.astype(np.float32))

X_wraa = torch.tensor(X_wraa)
X_rlr  = torch.tensor(X_rlr)
X_bsr  = torch.tensor(X_bsr)

y_wraa = torch.tensor(y_wraa_scaled).view(-1, 1)
y_bsr  = torch.tensor(y_bsr_scaled).view(-1, 1)
y_rlr  = torch.tensor(y_rlr_scaled).view(-1, 1)

Train wRAA prediction neural network

In [21]:
wraa_model = nn.Sequential(
    nn.Linear(5, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(wraa_model.parameters(), lr=0.01)
for epoch in range(300):
    preds = wraa_model(X_wraa)
    loss = loss_fn(preds, y_wraa)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("wRAA", epoch, loss.item())

wRAA 0 0.9994096755981445
wRAA 50 0.7470129728317261
wRAA 100 0.717595636844635
wRAA 150 0.711247444152832
wRAA 200 0.7086370587348938
wRAA 250 0.7075719237327576


Train BSR prediction for neural network

In [22]:
bsr_model = nn.Sequential(
    nn.Linear(3, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(bsr_model.parameters(), lr=0.01)
for epoch in range(300):
    preds = bsr_model(X_bsr)
    loss = loss_fn(preds, y_bsr)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("BsR", epoch, loss.item())

BsR 0 1.0309042930603027
BsR 50 0.18194495141506195
BsR 100 0.17346841096878052
BsR 150 0.17236754298210144
BsR 200 0.17182810604572296
BsR 250 0.17120122909545898


Train RLR prediction for nerual network

In [23]:
rlr_model = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(rlr_model.parameters(), lr=0.01)
for epoch in range(300):
    preds = rlr_model(X_rlr)
    loss = loss_fn(preds, y_rlr)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("RLR", epoch, loss.item())

RLR 0 1.0112446546554565
RLR 50 0.5676611661911011
RLR 100 0.5406180620193481
RLR 150 0.5240145921707153
RLR 200 0.5030254125595093
RLR 250 0.4841744303703308


Save neural netowrk models

In [24]:

torch.save(wraa_model.state_dict(), '../model/wraa_model.pth')
torch.save(bsr_model.state_dict(), '../model/bsr_model.pth') 
torch.save(rlr_model.state_dict(), '../model/rlr_model.pth')


Evaluate performence of random forest

In [25]:
wraa_preds_rf = rf_wraa.predict(X_wraa_test)
rf_wraa_r2 = r2_score(y_test["wRAA"], wraa_preds_rf)
rlr_preds_rf = rf_rlr.predict(X_rlr_test)
rf_rlr_r2 = r2_score(y_test["RLR"], rlr_preds_rf)
bsr_preds_rf = rf_bsr.predict(X_bsr_test)
rf_bsr_r2 = r2_score(y_test["BsR"], bsr_preds_rf)
print("Random Forest R2 Scores:")
print("wRAA R2:", rf_wraa_r2)   
print("RLR R2:", rf_rlr_r2)
print("BsR R2:", rf_bsr_r2)

Random Forest R2 Scores:
wRAA R2: 0.22744091021343094
RLR R2: 0.5889495906124795
BsR R2: 0.8599591227340375


Evaluate performence of nerual network

In [26]:
print("\nNeural Network R2 Scores:")
X_wraa_test = torch.tensor(scaler_X_wraa.transform(X_wraa_test.values.astype(np.float32)))
X_rlr_test  = torch.tensor(scaler_X_rlr.transform(X_rlr_test.values.astype(np.float32)))
X_bsr_test  = torch.tensor(scaler_X_bsr.transform(X_bsr_test.values.astype(np.float32)))

wraa_model.eval()
with torch.no_grad():
    preds = wraa_model(X_wraa_test).numpy()
    preds = scaler_y_wraa.inverse_transform(preds).flatten()  # ← unscale
    y_true = y_test["wRAA"].values
    print("wRAA R2:", r2_score(y_true, preds))

rlr_model.eval()
with torch.no_grad():
    preds = rlr_model(X_rlr_test).numpy()
    preds = scaler_y_rlr.inverse_transform(preds).flatten()   # ← unscale
    y_true = y_test["RLR"].values
    print("RLR R2:", r2_score(y_true, preds))

bsr_model.eval()
with torch.no_grad():
    preds = bsr_model(X_bsr_test).numpy()
    preds = scaler_y_bsr.inverse_transform(preds).flatten()   # ← unscale
    y_true = y_test["BsR"].values
    print("BsR R2:", r2_score(y_true, preds))


Neural Network R2 Scores:
wRAA R2: 0.2717654415420493
RLR R2: 0.5268473486292222
BsR R2: 0.8422308304696061
